In [ ]:
import os
import time
import subprocess
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pennylane as qml
from dataclasses import dataclass


@dataclass
class QMLConfig:
    # --- Basic settings ---
    seed: int = 42
    n_qubits: int = 4
    n_layers: int = 5
    n_classes: int = 4
    data_path: str = "./PAC/processed_data_multiclass.pt"
    
    # Directory to save results
    save_dir: str = "./results"
    
    # Set to -1 to enable automatic GPU selection
    device_id: int = 0  
    
    # --- Training settings ---
    batch_size: int = 64
    learning_rate: float = 0.005
    num_epochs: int = 150
    
    # --- Data sampling ---
    n_train_limit: int = 1000  
    n_test_limit: int = 200    
    
    # --- Adversarial attack parameters ---
    train_epsilon: float = 0.03
    train_alpha: float = 0.01
    train_iter: int = 3
    
    test_epsilon: float = 0.05
    test_alpha: float = 0.01
    test_iter: int = 10


# ==========================================
# 2. Core Trainer Class
# ==========================================
class QuantumAdversarialTrainer:
    def __init__(self, config: QMLConfig):
        self.cfg = config
        
        # 1. First set the seed (critical!)
        self._set_seed()
        
        self.device = self._init_device()
        self.qnode = self._build_qnode()
        self.model = self._build_model().to(self.device)
        self.criterion = nn.CrossEntropyLoss()
        
    def _set_seed(self):
        """Fix all random sources to ensure consistency of parameter initialisation and data sampling."""
        seed = self.cfg.seed
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # if using multi-GPU
        # Ensure determinism for convolution etc.
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        print(f"Global random seed locked: {seed}")

    def _init_device(self):
        target_id = self.cfg.device_id
        if target_id == -1:
            try:
                cmd = "nvidia-smi --query-gpu=memory.free --format=csv,nounits,noheader"
                mem_info = subprocess.check_output(cmd.split()).decode('ascii').strip().split('\n')
                memory_free_values = [int(x) for x in mem_info]
                target_id = int(np.argmax(memory_free_values))
                print(f"Auto-selected best GPU: GPU {target_id} (free {max(memory_free_values)} MB)")
            except Exception as e:
                print(f"Auto-selection failed ({e}), falling back to GPU 0")
                target_id = 0
        
        os.environ["CUDA_VISIBLE_DEVICES"] = str(target_id)
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    def _build_qnode(self):
        try:
            dev = qml.device("lightning.gpu", wires=self.cfg.n_qubits)
        except:
            print("Using CPU simulation")
            dev = qml.device("default.qubit", wires=self.cfg.n_qubits)

        @qml.qnode(dev, interface='torch', diff_method='adjoint')
        def circuit(inputs, weights):
            # ========================================================
            # Modification: Data Re-uploading structure
            # Logic: loop over n_layers times, re-encode data each layer, then apply an entangling layer
            # ========================================================
            for i in range(self.cfg.n_layers):
                # 1. Data Encoding (Re-uploading)
                qml.templates.AngleEmbedding(inputs, wires=range(self.cfg.n_qubits))
                
                # 2. Entangling Layer
                # Use slice weights[i:i+1] to keep dimension (1, n_qubits, 3)
                # so StronglyEntanglingLayers knows it is a single layer
                layer_weights = weights[i:i+1]
                qml.templates.StronglyEntanglingLayers(layer_weights, wires=range(self.cfg.n_qubits))
            
            return [qml.expval(qml.PauliZ(i)) for i in range(self.cfg.n_qubits)]
        
        return circuit

    def _build_model(self):
        class PureQuantumNet(nn.Module):
            def __init__(self, q_node, n_layers, n_qubits):
                super().__init__()
                # The weight definition remains unchanged, providing a total of n_layers parameter sets
                weight_shapes = {"weights": (n_layers, n_qubits, 3)}
                self.q_layer = qml.qnn.TorchLayer(q_node, weight_shapes)
                self.fixed_scale = 10.0 

            def forward(self, x):
                q_out = self.q_layer(x)
                return q_out * self.fixed_scale
        
        return PureQuantumNet(self.qnode, self.cfg.n_layers, self.cfg.n_qubits)

    def pgd_attack(self, inputs, labels, epsilon, alpha, num_iter):
        delta = torch.zeros_like(inputs, requires_grad=True).to(self.device)
        with torch.no_grad():
            delta.uniform_(-epsilon, epsilon)
            delta.add_(inputs).clamp_(0, np.pi).sub_(inputs)
        
        # Freeze model parameters (weights), only differentiate with respect to delta
        # Even with Data Re-uploading, parameters are still in weights, so logic remains unchanged
        for param in self.model.parameters(): param.requires_grad = False

        for _ in range(num_iter):
            adv_inputs = inputs + delta
            outputs = self.model(adv_inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            with torch.no_grad():
                grad = delta.grad.detach()
                delta.add_(alpha * grad.sign())
                delta.clamp_(-epsilon, epsilon)
                delta.add_(inputs).clamp_(0, np.pi).sub_(inputs)
                delta.grad.zero_()

        for param in self.model.parameters(): param.requires_grad = True
        return (inputs + delta).detach()

    def load_data(self):
        if not os.path.exists(self.cfg.data_path):
            raise FileNotFoundError(f"Data file not found: {self.cfg.data_path}")
        
        print(f"Loading data: {self.cfg.data_path}")
        data = torch.load(self.cfg.data_path)
        
        X_train_full, y_train_full = data["X_train"], data["y_train"]
        indices_train = torch.randperm(len(X_train_full))
        limit_train = min(self.cfg.n_train_limit, len(X_train_full))
        X_train = X_train_full[indices_train[:limit_train]]
        y_train = y_train_full[indices_train[:limit_train]]

        X_test_full, y_test_full = data["X_test"], data["y_test"]
        indices_test = torch.randperm(len(X_test_full))
        limit_test = min(self.cfg.n_test_limit, len(X_test_full))
        X_test = X_test_full[indices_test[:limit_test]]
        y_test = y_test_full[indices_test[:limit_test]]

        print(f"   Train: {len(y_train)} | Test: {len(y_test)}")

        return (
            DataLoader(TensorDataset(X_train, y_train), batch_size=self.cfg.batch_size, shuffle=True, drop_last=True),
            DataLoader(TensorDataset(X_test, y_test), batch_size=self.cfg.batch_size, shuffle=False)
        )

    def evaluate(self, loader, epsilon, alpha, num_iter, max_batches=None):
        self.model.eval()
        running_loss, correct, total = 0.0, 0, 0
        iterator = iter(loader)
        limit = len(loader) if max_batches is None else min(max_batches, len(loader))
        
        for _ in range(limit):
            try: inputs, labels = next(iterator)
            except StopIteration: break
            
            inputs, labels = inputs.to(self.device), labels.to(self.device)
            adv_inputs = self.pgd_attack(inputs, labels, epsilon, alpha, num_iter)
            
            with torch.no_grad():
                outputs = self.model(adv_inputs)
                loss = self.criterion(outputs, labels)
                running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
                
        if total == 0: return 0.0, 0.0
        return running_loss / total, 100 * correct / total

    def run(self):
        # Slightly modified filename to reflect architecture change (optional)
        file_name = f"ReUpload_layers{self.cfg.n_layers}_data{self.cfg.n_train_limit}_eps{self.cfg.train_epsilon}.pth"
        full_save_path = os.path.join(self.cfg.save_dir, file_name)
        
        if not os.path.exists(self.cfg.save_dir):
            os.makedirs(self.cfg.save_dir)

        train_loader, test_loader = self.load_data()
        optimizer = optim.SGD(self.model.parameters(), lr=self.cfg.learning_rate)
        
        print(f"Start training (Data Re-uploading) | Epochs: {self.cfg.num_epochs} | Seed: {self.cfg.seed}")
        print(f"Results will be saved to: {full_save_path}")
        
        header = f"{'Epoch':<6} | {'Tr_Loss':<10} | {'Tr_Acc':<10} | {'Te_Loss':<10} | {'Te_Acc':<10}"
        print("-" * len(header))
        print(header)
        print("-" * len(header))

        start_time = time.time()
        
        history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
        
        for epoch in range(self.cfg.num_epochs):
            self.model.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                
                adv_inputs = self.pgd_attack(inputs, labels, 
                                           self.cfg.train_epsilon, 
                                           self.cfg.train_alpha, 
                                           self.cfg.train_iter)
                
                optimizer.zero_grad()
                outputs = self.model(adv_inputs)
                loss = self.criterion(outputs, labels)
                loss.backward()
                optimizer.step()
            
            tr_loss, tr_acc = self.evaluate(train_loader, self.cfg.train_epsilon, self.cfg.train_alpha, self.cfg.train_iter, max_batches=5)
            te_loss, te_acc = self.evaluate(test_loader, self.cfg.test_epsilon, self.cfg.test_alpha, self.cfg.test_iter)
            
            history['train_loss'].append(tr_loss)
            history['train_acc'].append(tr_acc)
            history['test_loss'].append(te_loss)
            history['test_acc'].append(te_acc)
            
            print(f"{epoch+1:<6} | {tr_loss:.4f}{' ':>4} | {tr_acc:.2f}%{' ':>4} | {te_loss:.4f}{' ':>4} | {te_acc:.2f}%")
            
        total_time = time.time() - start_time
            
        save_dict = {
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'config': self.cfg,
            'history': history,
            'total_time': total_time
        }
        
        torch.save(save_dict, full_save_path)
        print("-" * len(header))
        print(f"Training completed, results saved to: {full_save_path}")
        
        return self.model